# Robust implementation of the softmax operation

This exemplifies the pitfalls one can encounter when implementing the softmax operation (which maps model outputs into probabilities).

We import the SciPy softmax as a comparison.

In [1]:
import numpy as np
from scipy.special import softmax

Here is our naive softmax implementation, simply implementing the regular softmax formula
$$
P(y_i | x_i) = \frac{e^{s_i}}{\sum_{j=1}^N e^{s_j}}
$$
where $s_i$ is the output from some model (e.g. a linear classifier).

In [2]:
def bad_softmax(scores):
    expscores = np.exp(scores)
    return expscores / expscores.sum()

To show how this softmax implementation behaves, we apply it to a vector of scores that are large enough to cause the `exp` call to overflow. Similarly, if you call it on a set of large negative values, you get underflow problems.

The SciPy implementation on the other hand does not have a problem with this.

In [3]:
a = np.array([800, 801, 802])

softmax(a), bad_softmax(a)

/var/folders/h3/xqjxfsm14t5f_nny2zq81_640000gp/T/ipykernel_68740/579640602.py:2: RuntimeWarning: overflow encountered in exp
  expscores = np.exp(scores)
/var/folders/h3/xqjxfsm14t5f_nny2zq81_640000gp/T/ipykernel_68740/579640602.py:3: RuntimeWarning: invalid value encountered in divide
  return expscores / expscores.sum()


(array([0.09003057, 0.24472847, 0.66524096]), array([nan, nan, nan]))

The following function shows how to get around the overflow/underflow problems. See [this document](https://www.cse.chalmers.se/~richajo/dit866/files/softmax_logsumexp.pdf) for an explanation.

In [4]:
def good_softmax(scores):
    m = np.max(scores)
    safe_scores = scores - m
    expscores = np.exp(safe_scores)
    return np.exp(safe_scores - np.log(expscores.sum()))